# RRT Demo: Fast Feasible Path Search

This notebook teaches RRT (Rapidly-exploring Random Tree) as a practical first-solution planner.

What you will learn:
- how random sampling grows a search tree from the start,
- how step size and goal bias change exploration behavior,
- why RRT is often fast but not cost-optimal.

How to use this notebook:
1. Run cells in order.
2. Inspect tree shape before judging path quality.
3. Experiment with `goal_bias`, `step_len`, and `max_iters`.

In [ ]:
%matplotlib inline\n
\n
import math\n
import random\n
\n
import matplotlib.pyplot as plt\n
import numpy as np

## 1. Build the map and define start/goal

We create an occupancy grid with narrow passages to make random exploration meaningful.

In [ ]:
H, W = 60, 60\n
grid = np.zeros((H, W), dtype=np.uint8)\n
\n
grid[8:52, 22] = 1\n
grid[8:52, 38] = 1\n
grid[30, 22:39] = 1\n
grid[16, 22] = 0\n
grid[44, 38] = 0\n
grid[30, 30] = 0\n
\n
start = (5.0, 5.0)\n
goal = (55.0, 55.0)

## 2. Geometry utilities and collision checks

RRT repeatedly proposes short edges, so reliable edge collision tests are essential.

In [ ]:
def in_bounds(r, c):\n
    return 0 <= r < H and 0 <= c < W\n
\n
def free_point(p):\n
    r, c = int(round(p[0])), int(round(p[1]))\n
    return in_bounds(r, c) and grid[r, c] == 0\n
\n
def dist(a, b):\n
    return math.hypot(a[0] - b[0], a[1] - b[1])\n
\n
def line_free(a, b):\n
    n = int(max(abs(b[0] - a[0]), abs(b[1] - a[1]))) + 1\n
    for i in range(n + 1):\n
        t = i / max(1, n)\n
        p = (a[0] + t * (b[0] - a[0]), a[1] + t * (b[1] - a[1]))\n
        if not free_point(p):\n
            return False\n
    return True

## 3. Run the RRT growth loop

This is the core algorithm loop.
- sample random targets (with occasional goal bias),
- extend from nearest node by one step,
- keep only collision-free additions.

Observe: a larger goal bias can speed up reaching the goal, but may reduce broad exploration.

In [ ]:
rng = random.Random(13)\n
step_len = 2.0\n
goal_radius = 2.5\n
goal_bias = 0.12\n
max_iters = 3000\n
\n
nodes = [start]\n
parent = {0: None}\n
goal_idx = None\n
\n
for _ in range(max_iters):\n
    if rng.random() < goal_bias:\n
        sample = goal\n
    else:\n
        sample = (rng.uniform(0, H - 1), rng.uniform(0, W - 1))\n
\n
    near_i = min(range(len(nodes)), key=lambda i: dist(nodes[i], sample))\n
    near = nodes[near_i]\n
\n
    d = dist(near, sample)\n
    if d < 1e-9:\n
        continue\n
\n
    t = min(step_len / d, 1.0)\n
    new = (near[0] + t * (sample[0] - near[0]), near[1] + t * (sample[1] - near[1]))\n
\n
    if not free_point(new):\n
        continue\n
    if not line_free(near, new):\n
        continue\n
\n
    nodes.append(new)\n
    ni = len(nodes) - 1\n
    parent[ni] = near_i\n
\n
    if dist(new, goal) <= goal_radius and line_free(new, goal):\n
        nodes.append(goal)\n
        gi = len(nodes) - 1\n
        parent[gi] = ni\n
        goal_idx = gi\n
        break\n
\n
print('Nodes in tree:', len(nodes), 'Found path:', goal_idx is not None)

## 4. Extract and inspect the found path

If `goal_idx` is connected, we backtrack using the parent pointers to recover the route.

In [ ]:
path = []\n
if goal_idx is not None:\n
    cur = goal_idx\n
    while cur is not None:\n
        path.append(nodes[cur])\n
        cur = parent[cur]\n
    path.reverse()\n
\n
print('Path points:', len(path))

## 5. Visual interpretation and self-check

Read the final plot with these questions:
- Is the tree spread balanced or strongly biased?
- Does the path look jagged (common for plain RRT)?
- What changes if you halve or double `step_len`?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))\n
ax.imshow(grid, cmap='Greys', origin='upper')\n
\n
for i in range(1, len(nodes)):\n
    pi = parent[i]\n
    if pi is None:\n
        continue\n
    a, b = nodes[pi], nodes[i]\n
    ax.plot([a[1], b[1]], [a[0], b[0]], color='lightsteelblue', linewidth=0.6)\n
\n
if path:\n
    xs = [p[1] for p in path]\n
    ys = [p[0] for p in path]\n
    ax.plot(xs, ys, color='deepskyblue', linewidth=2.4, label='RRT path')\n
\n
ax.scatter(start[1], start[0], c='limegreen', s=90, marker='o', label='start')\n
ax.scatter(goal[1], goal[0], c='crimson', s=90, marker='*', label='goal')\n
ax.set_title('RRT tree growth and feasible path')\n
ax.set_xticks([])\n
ax.set_yticks([])\n
ax.legend(loc='upper right')\n
plt.tight_layout()\n
plt.show()